In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vit import TimeSformer
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-22 01:28:04,856]::3184030073::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,fullshot,tagdata,transform=None):
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120

# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [9]:
train_set_data = ChalearnDataset(
    fullshot=train_set_data[1],
    tagdata=train_set_data[2],
    transform=transform
)

val_set_data = ChalearnDataset(
    fullshot=validation_set_data[1],
    tagdata=validation_set_data[2],
    transform=transform
)

print(1)


1


In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# Multi-seed Training
# ============================================================

# Only ONE checkpoint file is kept.
# Whenever a better validation MSE is found across all seeds,
# this same file is overwritten.
# from full_30s_divided_by_person_SCENE3.vivit_vst_timsformer336699.TimeSformer_with_r2plus1d_1.timesformer.models.vit import TimeSformer


best_model_path = "./save_timesformer_folder/model_best_multiseed.pth"

seed_results = []
global_best_mse = float("inf")
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(0):

    for seed in SEEDS:

        print("\n" + "=" * 80)
        print(f"Start training | seed = {seed}")
        print("=" * 80)

        # ----------------------------------------------------
        # Reproducibility
        # ----------------------------------------------------
        set_seed(seed)

        # Re-create DataLoaders for each seed so that the
        # shuffled training order is also controlled by seed.
        generator = torch.Generator()
        generator.manual_seed(seed)

        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # Every seed starts from a newly initialized model.
        # ----------------------------------------------------
        model=TimeSformer()#num_frames=frame_nums,batchsz = batchsz
        model.to(device)

        criterion = torch.nn.MSELoss().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

        seed_best_mse = float("inf")
        seed_best_epoch = -1
        no_improve_count = 0

        for i in range(start_epoch, epochs):

            train_avg_loss = 0
            val_avg_loss = 0

            train_avg_o_loss = 0
            train_avg_c_loss = 0
            train_avg_e_loss = 0
            train_avg_a_loss = 0
            train_avg_n_loss = 0

            val_avg_o_loss = 0
            val_avg_c_loss = 0
            val_avg_e_loss = 0
            val_avg_a_loss = 0
            val_avg_n_loss = 0

            # ------------------------------------------------
            # Train
            # ------------------------------------------------
            model.train()

            for fullshot, big_five_data in tqdm(
                train_dataloader,
                desc=f"Seed {seed} | Train {i + 1}/{epochs}"
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1)
                big_five_data = big_five_data.squeeze()
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)

                optimizer.zero_grad()

                hypothesis = model(fullshot)
                loss = criterion(hypothesis, big_five_data)

                loss.backward()
                optimizer.step()

                train_avg_loss += loss

            train_avg_loss = train_avg_loss / len(train_dataloader)

            # ------------------------------------------------
            # Validation
            # ------------------------------------------------
            model.eval()

            with torch.no_grad():

                for fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc=f"Seed {seed} | Valid {i + 1}/{epochs}"
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1)
                    big_five_data = big_five_data.squeeze()

                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    hypothesis = model(fullshot)
                    val_loss = criterion(hypothesis, big_five_data)

                    hypothesiso, hypothesisc, hypothesise, hypothesisa, hypothesisn = \
                        extract_ocean(hypothesis, len(fullshot))

                    big_five_datao, big_five_datac, big_five_datae, \
                        big_five_dataa, big_five_datan = \
                        extract_ocean(big_five_data, len(fullshot))

                    val_o_loss = criterion(hypothesiso, big_five_datao)
                    val_c_loss = criterion(hypothesisc, big_five_datac)
                    val_e_loss = criterion(hypothesise, big_five_datae)
                    val_a_loss = criterion(hypothesisa, big_five_dataa)
                    val_n_loss = criterion(hypothesisn, big_five_datan)

                    val_avg_loss += val_loss
                    val_avg_o_loss += val_o_loss
                    val_avg_c_loss += val_c_loss
                    val_avg_e_loss += val_e_loss
                    val_avg_a_loss += val_a_loss
                    val_avg_n_loss += val_n_loss

                val_avg_loss = val_avg_loss / len(val_dataloader)
                val_avg_o_loss = val_avg_o_loss / len(val_dataloader)
                val_avg_c_loss = val_avg_c_loss / len(val_dataloader)
                val_avg_e_loss = val_avg_e_loss / len(val_dataloader)
                val_avg_a_loss = val_avg_a_loss / len(val_dataloader)
                val_avg_n_loss = val_avg_n_loss / len(val_dataloader)

            # Convert to Python float for comparisons / logging.
            current_val_mse = val_avg_loss.item()

            # ------------------------------------------------
            # Best result of the CURRENT seed
            # ------------------------------------------------
            if current_val_mse < seed_best_mse:
                seed_best_mse = current_val_mse
                seed_best_epoch = i + 1
                no_improve_count = 0
            else:
                no_improve_count += 1

            # ------------------------------------------------
            # Global best checkpoint across ALL seeds.
            # ------------------------------------------------
            if current_val_mse < global_best_mse:
                global_best_mse = current_val_mse
                global_best_seed = seed
                global_best_epoch = i + 1

                torch.save({
                    "seed": seed,
                    "epoch": i + 1,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "loss": current_val_mse,
                }, best_model_path)

                logger.info(
                    "Global best updated | seed: {} | epoch: {} | MSE: {:.6f}".format(
                        seed, i + 1, current_val_mse
                    )
                )

            logger.debug(
                "Seed: {} | Epoch: {} | MSE: {:.6f} | "
                "O-MSE: {:.6f} | C-MSE: {:.6f} | E-MSE: {:.6f} | "
                "A-MSE: {:.6f} | N-MSE: {:.6f}".format(
                    seed, i + 1, current_val_mse,
                    val_avg_o_loss.item(),
                    val_avg_c_loss.item(),
                    val_avg_e_loss.item(),
                    val_avg_a_loss.item(),
                    val_avg_n_loss.item()
                )
            )

            torch.cuda.empty_cache()

            # ------------------------------------------------
            # Early stopping for the CURRENT seed.
            # Stop when validation MSE has not improved for
            # `patience` consecutive epochs.
            # ------------------------------------------------
            if no_improve_count >= patience:
                logger.info(
                    "Early stopping | seed: {} | epoch: {} | "
                    "best epoch: {} | best MSE: {:.6f}".format(
                        seed, i + 1, seed_best_epoch, seed_best_mse
                    )
                )
                break

        # ----------------------------------------------------
        # Store only scalar results for this seed.
        # ----------------------------------------------------
        seed_results.append({
            "seed": seed,
            "best_epoch": seed_best_epoch,
            "best_mse": seed_best_mse
        })

        print(
            f"Seed {seed} finished | "
            f"Best epoch = {seed_best_epoch} | "
            f"Best MSE = {seed_best_mse:.6f}"
        )

        # ----------------------------------------------------
        # Release memory before starting the next seed.
        # ----------------------------------------------------
        del model
        del optimizer
        del criterion
        del train_dataloader
        del val_dataloader
        del generator

        gc.collect()
        torch.cuda.empty_cache()


# ============================================================
# Multi-seed Summary
# ============================================================

all_mse = np.array(
    [result["best_mse"] for result in seed_results],
    dtype=np.float64
)

mean_mse = all_mse.mean()
std_mse = all_mse.std(ddof=1) if len(all_mse) > 1 else 0.0

print("\n" + "=" * 80)
print("MULTI-SEED RESULTS")
print("=" * 80)

for result in seed_results:
    print(
        "Seed {:>4} | Best Epoch {:>3} | MSE {:.6f}".format(
            result["seed"],
            result["best_epoch"],
            result["best_mse"]
        )
    )

print("-" * 80)
print(f"Mean MSE : {mean_mse:.6f}")
print(f"Std  MSE : {std_mse:.6f}")
print(
    f"Global best: seed={global_best_seed}, "
    f"epoch={global_best_epoch}, "
    f"MSE={global_best_mse:.6f}"
)
print(f"Saved checkpoint: {best_model_path}")
print("=" * 80)



Start training | seed = 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 42 | Train 1/120: 100%|██████████| 1091/1091 [04:19<00:00,  4.20it/s]
Seed 42 | Valid 1/120: 100%|██████████| 191/191 [00:17<00:00, 10.76it/s]
[2026-09-22 01:32:55,654]::4262348031::INFO::Global best updated | seed: 42 | epoch: 1 | MSE: 1.408047
[2026-09-22 01:32:55,657]::4262348031::DEBUG::Seed: 42 | Epoch: 1 | MSE: 1.408047 | O-MSE: 1.306091 | C-MSE: 1.248206 | E-MSE: 1.143600 | A-MSE: 1.741005 | N-MSE: 1.601329
Seed 42 | Train 2/120: 100%|██████████| 1091/1091 [04:19<00:00,  4.20it/s]
Seed 42 | Valid 2/120: 100%|██████████| 191/191 [00:17<00:00, 10.95it/s]
[2026-09-22 01:37:37,181]::4262348031::INFO::Global best updated | seed: 42 | epoch: 2 | MSE: 1.289204
[2026-09-22 01:37:37,182]::4262348031::DEBUG::Seed: 42 | Epoch: 2 | MSE: 1.289204 | O-MSE: 1.023290 | C-MSE: 1.264022 | E-MSE: 1.167948 | A-MSE: 1.476271 | N-MSE: 1.514488
Seed 42 | Train 3/120: 100%|██████████| 1091/1091 [04:20<00:00,  4.19it/s]
Seed 42 | Valid 3/120: 100%|██████████| 191/191 [00:18<00:00, 10.60it/s]
[2026-

Seed 42 finished | Best epoch = 2 | Best MSE = 1.289204

Start training | seed = 123
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 123 | Train 1/120: 100%|██████████| 1091/1091 [04:22<00:00,  4.15it/s]
Seed 123 | Valid 1/120: 100%|██████████| 191/191 [00:18<00:00, 10.51it/s]
[2026-09-22 03:15:45,494]::4262348031::DEBUG::Seed: 123 | Epoch: 1 | MSE: 1.434590 | O-MSE: 1.313721 | C-MSE: 1.369589 | E-MSE: 1.196845 | A-MSE: 1.741792 | N-MSE: 1.551001
Seed 123 | Train 2/120: 100%|██████████| 1091/1091 [04:22<00:00,  4.16it/s]
Seed 123 | Valid 2/120: 100%|██████████| 191/191 [00:18<00:00, 10.45it/s]
[2026-09-22 03:20:27,462]::4262348031::DEBUG::Seed: 123 | Epoch: 2 | MSE: 1.390916 | O-MSE: 1.300684 | C-MSE: 1.293090 | E-MSE: 1.277096 | A-MSE: 1.689113 | N-MSE: 1.394601
Seed 123 | Train 3/120: 100%|██████████| 1091/1091 [04:22<00:00,  4.15it/s]
Seed 123 | Valid 3/120: 100%|██████████| 191/191 [00:18<00:00, 10.40it/s]
[2026-09-22 03:25:09,828]::4262348031::DEBUG::Seed: 123 | Epoch: 3 | MSE: 1.305799 | O-MSE: 1.100337 | C-MSE: 1.270325 | E-MSE: 1.158304 | A-MSE: 1.556958 | N-MSE: 1.443068
Seed 123 | Train 4/120: 100%|██

Seed 123 finished | Best epoch = 5 | Best MSE = 1.291630

Start training | seed = 777
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


...

Seed 777 finished | Best epoch = 2 | Best MSE = 1.341760

Start training | seed = 1004
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 1004 | Train 1/120: 100%|██████████| 1091/1091 [04:22<00:00,  4.16it/s]
Seed 1004 | Valid 1/120: 100%|██████████| 191/191 [00:18<00:00, 10.36it/s]
[2026-09-22 06:57:11,908]::4262348031::DEBUG::Seed: 1004 | Epoch: 1 | MSE: 1.677207 | O-MSE: 1.586362 | C-MSE: 1.588227 | E-MSE: 1.217505 | A-MSE: 2.001144 | N-MSE: 1.992798
Seed 1004 | Train 2/120: 100%|██████████| 1091/1091 [04:22<00:00,  4.15it/s]
Seed 1004 | Valid 2/120: 100%|██████████| 191/191 [00:18<00:00, 10.36it/s]
[2026-09-22 07:01:54,467]::4262348031::DEBUG::Seed: 1004 | Epoch: 2 | MSE: 1.416111 | O-MSE: 0.937237 | C-MSE: 1.333552 | E-MSE: 1.346114 | A-MSE: 1.724300 | N-MSE: 1.739349
Seed 1004 | Train 3/120: 100%|██████████| 1091/1091 [04:22<00:00,  4.16it/s]
Seed 1004 | Valid 3/120: 100%|██████████| 191/191 [00:18<00:00, 10.28it/s]
[2026-09-22 07:06:36,893]::4262348031::DEBUG::Seed: 1004 | Epoch: 3 | MSE: 1.414939 | O-MSE: 0.927110 | C-MSE: 1.428743 | E-MSE: 1.228747 | A-MSE: 1.823738 | N-MSE: 1.666358
Seed 1004 | Train 4/12

Seed 1004 finished | Best epoch = 5 | Best MSE = 1.358590

Start training | seed = 2024
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


Seed 2024 | Train 1/120: 100%|██████████| 1091/1091 [04:22<00:00,  4.16it/s]
Seed 2024 | Valid 1/120: 100%|██████████| 191/191 [00:18<00:00, 10.37it/s]
[2026-09-22 08:54:51,812]::4262348031::DEBUG::Seed: 2024 | Epoch: 1 | MSE: 1.476905 | O-MSE: 1.373390 | C-MSE: 1.427494 | E-MSE: 1.152163 | A-MSE: 1.591932 | N-MSE: 1.839543
Seed 2024 | Train 2/120: 100%|██████████| 1091/1091 [04:21<00:00,  4.17it/s]
Seed 2024 | Valid 2/120: 100%|██████████| 191/191 [00:18<00:00, 10.41it/s]
[2026-09-22 08:59:33,332]::4262348031::DEBUG::Seed: 2024 | Epoch: 2 | MSE: 1.454193 | O-MSE: 1.238553 | C-MSE: 1.505206 | E-MSE: 1.396338 | A-MSE: 1.563724 | N-MSE: 1.567139
Seed 2024 | Train 3/120: 100%|██████████| 1091/1091 [04:21<00:00,  4.17it/s]
Seed 2024 | Valid 3/120: 100%|██████████| 191/191 [00:18<00:00, 10.41it/s]
[2026-09-22 09:04:14,755]::4262348031::DEBUG::Seed: 2024 | Epoch: 3 | MSE: 1.490017 | O-MSE: 1.135090 | C-MSE: 1.492530 | E-MSE: 1.537034 | A-MSE: 1.785920 | N-MSE: 1.499509
Seed 2024 | Train 4/12

Seed 2024 finished | Best epoch = 20 | Best MSE = 1.261257

MULTI-SEED RESULTS
Seed   42 | Best Epoch   2 | MSE 1.289204
Seed  123 | Best Epoch   5 | MSE 1.291630
Seed  777 | Best Epoch   2 | MSE 1.341760
Seed 1004 | Best Epoch   5 | MSE 1.358590
Seed 2024 | Best Epoch  20 | MSE 1.261257
--------------------------------------------------------------------------------
Mean MSE : 1.308488
Std  MSE : 0.040324
Global best: seed=2024, epoch=20, MSE=1.261257
Saved checkpoint: ./save_timesformer_folder/model_best_multiseed.pth
